# Updated data preparation — Snowflake to validated parquet

Based on `chunking_series_code_new.ipynb`. This notebook prepares data only; it does not train a model or fit an encoder.

**Run cells in order.** Review configuration first. Keep your existing Snowflake configuration module; no credentials are embedded here. Dependencies: pandas, numpy, pyarrow, snowflake-snowpark-python (with pandas support).

Changes: explicit feature lists; a consistent temporary Snowflake snapshot; deterministic dealer chunks; strict per-series date, target and static checks; calendar consistency across all selected series; collision checks for series keys; fresh output directories; hashes and a completion marker. No negative clipping, duplicate aggregation, missing-day imputation, or automatic file skipping.

The existing 52 festive columns and five time features are preserved. `IS_PRE_WEDDING_WINDOW` is computed from `MARRIAGE_DAY` for days −7 through −1. Overlaps remain binary. It is saved for inspection but **excluded from `future_covariates` by default** for the corrected baseline. Set `INCLUDE_PRE_WEDDING_WINDOW = True` for the separate wedding-feature experiment. No loss penalties are assigned here.

Dates and the valid-series filter are preserved from the uploaded notebook. They are not automatically moved to today's date. The filter may exclude series; this notebook reports that population rather than claiming to cover all business sales.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
import re
import sys
import uuid

import numpy as np
import pandas as pd

# Existing connection configuration; change this path if needed.
UTILS_DIR = Path(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
OUTPUT_ROOT = Path.cwd() / "prepared_data_updated"
TABLE_NAME = "MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES"
VALID_SERIES_TABLE = "MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING"
SNAPSHOT_SCHEMA = "MOP_DATABASE.SOQ"

TIME_COL = "CAL_DATE"
GROUP_COL = "PARENT_DEALER_CODE_MODEL_FAMILY"
DEALER_COL = "PARENT_DEALER_CODE"
TARGET_COL = "NET_SALES"
WEDDING_COL = "MARRIAGE_DAY"
PRE_WEDDING_COL = "IS_PRE_WEDDING_WINDOW"

TRAIN_START = "2023-04-01"
TRAIN_END = "2026-04-30"
VAL_START = "2026-05-01"
VAL_END = "2026-08-31"
FORECAST_START = "2026-09-01"
FORECAST_END = "2026-12-07"

# A smaller extraction chunk controls RAM; it does not change model batch size.
DEALERS_PER_CHUNK = 25
PRE_WEDDING_DAYS = 7
INCLUDE_PRE_WEDDING_WINDOW = False

STATIC_COLS = [
    "PARENT_DEALER_CODE", "MODEL_FAMILY", "MODEL_NAME", "BRAKE_TYPE",
    "IGNITION_TYPE", "WHEEL_TYPE", "COLOUR", "DEALER_CITY",
    "X_CITY_CATEGORY", "ZONAL_OFFICE_NAME",
]
# Explicit allowlist from the uploaded notebook: unknown columns are not
# silently admitted as known-future covariates (which could leak sales).
FESTIVE_COLS = [
    "HARTALIK_TEEJ", "GANESH_CHATURTHI", "JANMASHTAMI", "VISHWAKARMA_PUJA",
    "KARWA_CHAUTH", "ONAM",
    *[f"N{i}" for i in range(-16, 0)], "N",
    *[f"N+{i}" for i in range(1, 11)],
    "D-3", "D-2", "D-1", "D", *[f"D+{i}" for i in range(1, 7)],
    "HANUMAN_JAYANTI", "AKSHYA_TRITIYA", "BUDDHA_PURNIMA", "GANGA_DUSSEHRA",
    "JAGANNATH_RATHYATRA", "GURU_PURNIMA", "NAG_PANCHAMI",
    "RAKSHA_BANDHAN", "MARRIAGE_DAY",
]
TIME_FEATURES = [
    "DOW_SIN", "DOW_COS", "IS_MONTH_END", "IS_MONTH_START", "DAYS_TO_MONTH_END"
]

assert len(FESTIVE_COLS) == 52
assert len(set(STATIC_COLS + FESTIVE_COLS)) == len(STATIC_COLS + FESTIVE_COLS)
assert DEALERS_PER_CHUNK > 0 and PRE_WEDDING_DAYS > 0
assert pd.Timestamp(TRAIN_START) <= pd.Timestamp(TRAIN_END)
assert pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1) == pd.Timestamp(VAL_START)
assert pd.Timestamp(VAL_START) <= pd.Timestamp(VAL_END)
assert pd.Timestamp(VAL_END) + pd.Timedelta(days=1) == pd.Timestamp(FORECAST_START)
assert pd.Timestamp(FORECAST_START) <= pd.Timestamp(FORECAST_END)
CALENDAR_LOOKAHEAD_END = (
    pd.Timestamp(FORECAST_END) + pd.Timedelta(days=PRE_WEDDING_DAYS)
).strftime("%Y-%m-%d")

print(f"History: {TRAIN_START} to {VAL_END}; forecast: {FORECAST_START} to {FORECAST_END}")
print(f"MARRIAGE_DAY must be populated through {CALENDAR_LOOKAHEAD_END}.")
print("Pre-wedding feature included in model roles:", INCLUDE_PRE_WEDDING_WINDOW)


## Validation and feature functions

The pipeline expects one daily row per selected series throughout history and forecast, as in the uploaded data. Missing dates or a changing static attribute cause a clear error instead of silently shifting time or taking the first attribute value. New/retired series with intentionally shorter histories require an explicit lifecycle policy before relaxing these checks.

Historical `NET_SALES` must be finite, nonnegative and integer-valued for the planned count likelihood. If this fails, resolve the target definition; the notebook will not turn negative net sales into zero. Forecast targets are written as missing, never as observed sales.


In [ ]:
def qi(name):
    """Quote a single SQL identifier, including N-16 and D+1 columns."""
    return '"' + str(name).replace('"', '""') + '"'


def qualified(name):
    parts = name.split(".")
    if not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", p) for p in parts):
        raise ValueError(f"Use unquoted simple database/schema/table names: {name!r}")
    return ".".join(qi(p.upper()) for p in parts)


def atomic_json(value, path):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)


def atomic_parquet(frame, path):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temp, index=False, engine="pyarrow")
    temp.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def require_daily_dates(values, start, end, label):
    dates = pd.DatetimeIndex(pd.to_datetime(values, errors="raise"))
    expected = pd.date_range(start, end, freq="D")
    if dates.hasnans or dates.has_duplicates:
        raise ValueError(f"{label}: null or duplicate dates")
    if not dates.sort_values().equals(expected):
        missing = expected.difference(dates)
        extra = dates.difference(expected)
        raise ValueError(
            f"{label}: incorrect daily coverage. Missing {list(missing[:5])}; "
            f"extra {list(extra[:5])}"
        )


def pre_wedding_flag(calendar, wedding_calendar, days=7):
    """Wedding day excluded; a later wedding can still flag that day."""
    required_end = pd.Timestamp(calendar[TIME_COL].max()) + pd.Timedelta(days=days)
    require_daily_dates(
        wedding_calendar[TIME_COL], calendar[TIME_COL].min(), required_end,
        "Wedding calendar including lookahead",
    )
    flags = pd.to_numeric(wedding_calendar[WEDDING_COL], errors="raise")
    if flags.isna().any() or not flags.isin([0, 1]).all():
        raise ValueError(f"{WEDDING_COL} must be populated with binary 0/1 values")
    weddings = pd.DatetimeIndex(wedding_calendar.loc[flags.eq(1), TIME_COL])
    marked = pd.DatetimeIndex([])
    for offset in range(1, days + 1):
        marked = marked.union(weddings - pd.Timedelta(days=offset))
    return pd.to_datetime(calendar[TIME_COL]).isin(marked).astype("int8")


def engineer_calendar(calendar, wedding_calendar):
    cal = calendar.sort_values(TIME_COL).reset_index(drop=True).copy()
    require_daily_dates(cal[TIME_COL], TRAIN_START, FORECAST_END, "Shared calendar")
    for col in FESTIVE_COLS:
        values = pd.to_numeric(cal[col], errors="raise")
        if values.isna().any() or not values.isin([0, 1]).all():
            raise ValueError(f"Expected a binary festive flag: {col}")
        cal[col] = values.astype("float32")
    dt = cal[TIME_COL].dt
    cal["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7).astype("float32")
    cal["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7).astype("float32")
    cal["IS_MONTH_END"] = (dt.day >= dt.days_in_month - 2).astype("float32")
    cal["IS_MONTH_START"] = (dt.day <= 3).astype("float32")
    cal["DAYS_TO_MONTH_END"] = ((dt.days_in_month - dt.day) / 31).astype("float32")
    cal[PRE_WEDDING_COL] = pre_wedding_flag(cal, wedding_calendar, PRE_WEDDING_DAYS)
    return cal


def validate_chunk(frame, expected_groups, label):
    if frame.empty:
        raise ValueError(f"{label}: empty chunk")
    if frame[[TIME_COL, GROUP_COL] + STATIC_COLS].isna().any().any():
        raise ValueError(f"{label}: missing date, series key or static attribute")
    dates = pd.to_datetime(frame[TIME_COL], errors="raise")
    if not dates.eq(dates.dt.normalize()).all():
        raise ValueError(f"{label}: non-midnight timestamps; daily dates required")
    frame[TIME_COL] = dates
    frame[GROUP_COL] = frame[GROUP_COL].astype(str)
    if set(frame[GROUP_COL].unique()) != set(expected_groups):
        raise ValueError(f"{label}: unexpected/missing series")
    if frame.duplicated([GROUP_COL, TIME_COL]).any():
        raise ValueError(f"{label}: duplicate series/date rows")
    if not frame[TIME_COL].between(TRAIN_START, FORECAST_END).all():
        raise ValueError(f"{label}: dates outside configured range")

    groups = frame.groupby(GROUP_COL, sort=False, observed=True)
    # Unique daily dates + bounds + full row count proves continuity per series.
    stats = groups[TIME_COL].agg(["min", "max", "size"])
    required_n = len(pd.date_range(TRAIN_START, FORECAST_END))
    bad = stats[
        stats["min"].ne(pd.Timestamp(TRAIN_START)) |
        stats["max"].ne(pd.Timestamp(FORECAST_END)) |
        stats["size"].ne(required_n)
    ]
    if not bad.empty:
        raise ValueError(f"{label}: incomplete series histories/forecast rows:\n{bad.head()}")
    variation = groups[STATIC_COLS].nunique(dropna=False)
    if variation.gt(1).any().any():
        raise ValueError(f"{label}: static attributes change within series:\n"
                         f"{variation.loc[variation.gt(1).any(axis=1)].head()}")
    for col in STATIC_COLS:
        frame[col] = frame[col].astype(str)
        if frame[col].str.strip().eq("").any():
            raise ValueError(f"{label}: blank static attribute {col}")

    is_history = dates.le(pd.Timestamp(VAL_END))
    y = pd.to_numeric(frame.loc[is_history, TARGET_COL], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(y).all() or (y < 0).any() or not np.equal(y, np.floor(y)).all():
        raise ValueError(f"{label}: historical {TARGET_COL} must be nonnegative integer counts")
    if (y > 2**24).any():
        raise ValueError("Target exceeds exact float32 integer range; review target storage")
    frame[TARGET_COL] = np.nan
    frame.loc[is_history, TARGET_COL] = y
    frame[TARGET_COL] = frame[TARGET_COL].astype("float32")
    return frame.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)


## Snowflake extraction and calendar consistency

This creates a uniquely named **temporary table** in `SNAPSHOT_SCHEMA` containing the selected rows. All extracts use that snapshot so source updates cannot produce mismatched chunks. It does not update the source tables. Temporary storage and query execution use Snowflake resources; the temporary table is dropped in `finally`.

The shared calendar is checked against every selected series, not just the first dealer chunk. Regional calendar differences will stop execution; they require separate regional covariates instead of one shared calendar. Calendar dates through forecast end + seven days are required even if there are no weddings in that period: missing dates cannot safely be assumed to mean “no wedding”.


In [ ]:
def read_consistent_calendar(session, snapshot, columns, end):
    expressions = [qi(TIME_COL), "COUNT(*) AS N_ROWS"]
    for idx, col in enumerate(columns):
        expressions += [
            f"MIN({qi(col)}) AS LO_{idx}",
            f"MAX({qi(col)}) AS HI_{idx}",
            f"COUNT({qi(col)}) AS NN_{idx}",
        ]
    aggregate = session.sql(
        f"SELECT {', '.join(expressions)} FROM {snapshot} "
        f"WHERE {qi(TIME_COL)} BETWEEN TO_DATE(?) AND TO_DATE(?) "
        f"GROUP BY {qi(TIME_COL)} ORDER BY {qi(TIME_COL)}",
        params=[TRAIN_START, end],
    ).to_pandas()
    aggregate[TIME_COL] = pd.to_datetime(aggregate[TIME_COL])
    require_daily_dates(aggregate[TIME_COL], TRAIN_START, end, "Source calendar")
    result = aggregate[[TIME_COL]].copy()
    for idx, col in enumerate(columns):
        valid = (aggregate[f"NN_{idx}"].eq(aggregate["N_ROWS"]) &
                 aggregate[f"LO_{idx}"].eq(aggregate[f"HI_{idx}"]))
        if not valid.all():
            bad_dates = aggregate.loc[~valid, TIME_COL].head().tolist()
            raise ValueError(f"{col}: null or inconsistent across series on {bad_dates}")
        result[col] = aggregate[f"LO_{idx}"]
    return result


def prepare_data(session):
    # Dependency check before any expensive Snowflake work.
    import pyarrow

    source_columns = session.table(TABLE_NAME).columns
    required = list(dict.fromkeys([TIME_COL, GROUP_COL, TARGET_COL] + STATIC_COLS + FESTIVE_COLS))
    missing = sorted(set(required) - set(source_columns))
    if missing:
        raise ValueError(f"Source is missing required columns: {missing}")
    if GROUP_COL not in session.table(VALID_SERIES_TABLE).columns:
        raise ValueError(f"Valid-series table is missing {GROUP_COL}")

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    run_dir = OUTPUT_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    for folder in ["local_train_data", "local_test_data", "saved_group_keys"]:
        (run_dir / folder).mkdir()
    snapshot = qualified(SNAPSHOT_SCHEMA + ".TFT_PREP_" + uuid.uuid4().hex.upper())
    config = {
        "source_table": TABLE_NAME, "valid_series_table": VALID_SERIES_TABLE,
        "train_start": TRAIN_START, "train_end": TRAIN_END, "val_start": VAL_START,
        "val_end": VAL_END, "forecast_start": FORECAST_START, "forecast_end": FORECAST_END,
        "wedding_lookahead_end": CALENDAR_LOOKAHEAD_END,
        "pre_wedding_days": PRE_WEDDING_DAYS,
        "include_pre_wedding_window": INCLUDE_PRE_WEDDING_WINDOW,
        "static_columns": STATIC_COLS, "source_festive_columns": FESTIVE_COLS,
        "time_features": TIME_FEATURES, "dealers_per_chunk": DEALERS_PER_CHUNK,
        "target_policy": "nonnegative integer observed counts; no clipping",
        "date_policy": "complete daily grid for each selected series",
    }
    manifest = {"status": "running", "run_id": run_id, "config": config, "files": []}
    atomic_json(manifest, run_dir / "preparation_manifest.json")
    try:
        # Cast IDs at SQL boundary to preserve their categorical role regardless
        # of physical source type. Preserve string codes such as leading zeros.
        select_items = []
        for col in required:
            ref = "s." + qi(col)
            if col in STATIC_COLS or col == GROUP_COL:
                select_items.append(f"TO_VARCHAR({ref}) AS {qi(col)}")
            else:
                select_items.append(f"{ref} AS {qi(col)}")
        sql = (
            f"CREATE TEMPORARY TABLE {snapshot} AS SELECT {', '.join(select_items)} "
            f"FROM {qualified(TABLE_NAME)} s "
            f"WHERE s.{qi(TIME_COL)} BETWEEN TO_DATE(?) AND TO_DATE(?) "
            f"AND EXISTS (SELECT 1 FROM {qualified(VALID_SERIES_TABLE)} v "
            f"WHERE v.{qi(GROUP_COL)} = s.{qi(GROUP_COL)})"
        )
        print("Creating a consistent temporary Snowflake snapshot...", flush=True)
        session.sql(sql, params=[TRAIN_START, CALENDAR_LOOKAHEAD_END]).collect()

        calendar_source = read_consistent_calendar(session, snapshot, FESTIVE_COLS, FORECAST_END)
        weddings = read_consistent_calendar(session, snapshot, [WEDDING_COL], CALENDAR_LOOKAHEAD_END)
        calendar = engineer_calendar(calendar_source, weddings)
        print(f"Calendar checked across all selected series: {len(calendar):,} days", flush=True)

        mapping = session.sql(
            f"SELECT DISTINCT {qi(GROUP_COL)}, {qi(DEALER_COL)} FROM {snapshot} "
            f"WHERE {qi(TIME_COL)} <= TO_DATE(?)",
            params=[FORECAST_END],
        ).to_pandas()
        if mapping.empty or mapping.isna().any().any():
            raise ValueError("No selected series, or null group/dealer keys")
        if mapping[GROUP_COL].duplicated().any():
            raise ValueError("A series belongs to multiple dealers")
        mapping = mapping.sort_values([DEALER_COL, GROUP_COL]).reset_index(drop=True)
        # Preserve the previous key convention, but fail if replacement collides.
        mapping["PREPARED_GROUP_KEY"] = mapping[GROUP_COL].str.replace("<>", "_", regex=False)
        if mapping["PREPARED_GROUP_KEY"].duplicated().any():
            raise ValueError("Series-key sanitization collision; use a collision-safe key scheme")
        key_map = dict(zip(mapping[GROUP_COL], mapping["PREPARED_GROUP_KEY"]))
        dealers = sorted(mapping[DEALER_COL].unique().tolist())
        manifest.update({"series_count": len(mapping), "dealer_count": len(dealers)})
        print(f"Selected population: {len(mapping):,} series; {len(dealers):,} dealers", flush=True)

        def record(path, rows=None):
            item = {"path": str(path.relative_to(run_dir)), "sha256": sha256_file(path)}
            if rows is not None:
                item["rows"] = int(rows)
            manifest["files"].append(item)

        for name, frame in [("shared_calendar.parquet", calendar),
                            ("wedding_calendar.parquet", weddings),
                            ("series_key_mapping.parquet", mapping)]:
            path = run_dir / name
            atomic_parquet(frame, path)
            record(path, len(frame))
        group_path = run_dir / "saved_group_keys" / "group_keys.json"
        atomic_json(sorted(mapping["PREPARED_GROUP_KEY"].tolist()), group_path)
        record(group_path)

        summaries = []
        chunk_plan = []
        for chunk_idx, offset in enumerate(range(0, len(dealers), DEALERS_PER_CHUNK)):
            chunk_dealers = dealers[offset:offset + DEALERS_PER_CHUNK]
            expected_groups = mapping.loc[mapping[DEALER_COL].isin(chunk_dealers), GROUP_COL].tolist()
            # Festive values were already validated globally; join the small
            # canonical calendar locally rather than downloading repeated flags.
            columns = list(dict.fromkeys([TIME_COL, GROUP_COL, TARGET_COL] + STATIC_COLS))
            placeholders = ", ".join("?" for _ in chunk_dealers)
            frame = session.sql(
                f"SELECT {', '.join(qi(c) for c in columns)} FROM {snapshot} "
                f"WHERE {qi(TIME_COL)} <= TO_DATE(?) "
                f"AND {qi(DEALER_COL)} IN ({placeholders})",
                params=[FORECAST_END] + chunk_dealers,
            ).to_pandas()
            frame = validate_chunk(frame, expected_groups, f"Chunk {chunk_idx}")
            frame[GROUP_COL] = frame[GROUP_COL].map(key_map)
            frame = frame.merge(calendar, on=TIME_COL, how="left", validate="many_to_one")
            if frame[FESTIVE_COLS + TIME_FEATURES + [PRE_WEDDING_COL]].isna().any().any():
                raise ValueError("Calendar join produced missing covariates")
            historical = frame[TIME_COL].le(pd.Timestamp(VAL_END))
            for label, mask, folder in [
                ("history", historical, "local_train_data"),
                ("forecast", ~historical, "local_test_data"),
            ]:
                part = frame.loc[mask].copy()
                path = run_dir / folder / f"chunk_{chunk_idx:04d}.parquet"
                atomic_parquet(part, path)
                record(path, len(part))
                if label == "history":
                    summaries.append(part.groupby([TIME_COL, "MODEL_FAMILY", "MODEL_NAME"],
                                                   observed=True)[TARGET_COL].sum().reset_index())
                del part
            chunk_plan.append({"chunk": chunk_idx, "dealers": chunk_dealers,
                               "series": len(expected_groups)})
            atomic_json(manifest, run_dir / "preparation_manifest.json")
            print(f"Chunk {chunk_idx + 1}: {len(frame):,} rows validated and saved", flush=True)
            del frame
            gc.collect()

        audit = pd.concat(summaries, ignore_index=True).groupby(
            [TIME_COL, "MODEL_FAMILY", "MODEL_NAME"], observed=True
        )[TARGET_COL].sum().reset_index()
        audit_path = run_dir / "historical_sales_audit.parquet"
        atomic_parquet(audit, audit_path)
        record(audit_path, len(audit))
        plan_path = run_dir / "chunk_plan.json"
        atomic_json(chunk_plan, plan_path)
        record(plan_path)
        selected_festive = FESTIVE_COLS + ([PRE_WEDDING_COL] if INCLUDE_PRE_WEDDING_WINDOW else [])
        roles = {
            "static_covariates": STATIC_COLS, "festive_covariates": selected_festive,
            "calendar_covariates": TIME_FEATURES,
            "future_covariates": selected_festive + TIME_FEATURES,
            "available_extra_covariates": [] if INCLUDE_PRE_WEDDING_WINDOW else [PRE_WEDDING_COL],
            "train_start": TRAIN_START, "train_end": TRAIN_END,
            "val_start": VAL_START, "val_end": VAL_END,
            "forecast_start": FORECAST_START, "forecast_end": FORECAST_END,
            "time_col": TIME_COL, "group_col": GROUP_COL,
            "target_col": TARGET_COL, "freq": "D",
            "preparation_run_id": run_id,
        }
        roles_path = run_dir / "column_roles.json"
        atomic_json(roles, roles_path)
        record(roles_path)
        manifest["config_sha256"] = hashlib.sha256(
            json.dumps(config, sort_keys=True).encode("utf-8")
        ).hexdigest()
        manifest["packages"] = {
            name: importlib.metadata.version(name)
            for name in ["pandas", "numpy", "pyarrow", "snowflake-snowpark-python"]
        }
        manifest["historical_target_total"] = float(audit[TARGET_COL].sum())
        manifest["status"] = "complete"
        atomic_json(manifest, run_dir / "preparation_manifest.json")
        atomic_json({"manifest_sha256": sha256_file(run_dir / "preparation_manifest.json")},
                    run_dir / "PREPARATION_COMPLETE.json")
        print(f"\nCOMPLETE: {run_dir.resolve()}")
        print(f"Model future covariates: {len(roles['future_covariates'])}")
        return run_dir
    except Exception as exc:
        manifest.update(status="failed", error=str(exc))
        atomic_json(manifest, run_dir / "preparation_manifest.json")
        print(f"Preparation failed; do not train from this directory: {run_dir}")
        raise
    finally:
        try:
            session.sql(f"DROP TABLE IF EXISTS {snapshot}").collect()
        except Exception as cleanup_error:
            print("Temporary snapshot cleanup failed; close the session to release it:", cleanup_error)


## Local checks — no Snowflake connection

These small checks verify the wedding-window boundary, overlaps, lookahead requirement and basic rejection of invalid series data. They do not substitute for running the extraction against your Snowflake table.


In [ ]:
def run_local_checks():
    calendar = pd.DataFrame({TIME_COL: pd.date_range("2026-09-29", "2026-10-10")})
    wedding_calendar = pd.DataFrame({TIME_COL: pd.date_range("2026-09-29", "2026-10-17")})
    wedding_calendar[WEDDING_COL] = wedding_calendar[TIME_COL].eq(pd.Timestamp("2026-10-07")).astype(int)
    flags = pre_wedding_flag(calendar, wedding_calendar)
    actual = set(calendar.loc[flags.eq(1), TIME_COL])
    assert actual == set(pd.date_range("2026-09-30", "2026-10-06"))
    wedding_calendar.loc[wedding_calendar[TIME_COL].eq(pd.Timestamp("2026-10-10")), WEDDING_COL] = 1
    flags = pre_wedding_flag(calendar, wedding_calendar)
    assert flags.max() == 1
    assert flags.loc[calendar[TIME_COL].eq(pd.Timestamp("2026-10-07"))].iloc[0] == 1
    try:
        pre_wedding_flag(calendar, wedding_calendar.iloc[:-1])
    except ValueError:
        pass
    else:
        raise AssertionError("Incomplete lookahead was accepted")

    dates = pd.date_range(TRAIN_START, FORECAST_END)
    frame = pd.DataFrame({TIME_COL: dates, GROUP_COL: "test_series", TARGET_COL: 1.0})
    for col in STATIC_COLS:
        frame[col] = "test_category"
    good = validate_chunk(frame.copy(), ["test_series"], "Synthetic")
    assert good.loc[good[TIME_COL].gt(pd.Timestamp(VAL_END)), TARGET_COL].isna().all()
    broken = []
    broken.append(pd.concat([frame, frame.iloc[[0]]], ignore_index=True))
    broken.append(frame.iloc[1:].copy())
    negative = frame.copy(); negative.loc[0, TARGET_COL] = -1; broken.append(negative)
    changed_static = frame.copy(); changed_static.loc[0, "MODEL_FAMILY"] = "changed"; broken.append(changed_static)
    for case in broken:
        try:
            validate_chunk(case, ["test_series"], "Expected rejection")
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid series data was accepted")
    print("Local checks passed.")

run_local_checks()


## Run extraction

Reuses an existing `session` object if available; otherwise uses your existing `Snowflake_configuration.ds1_role_json`. An expired session must be reconnected normally. Re-running this cell creates a **new** run directory: it never resumes or silently mixes partial chunks.

The source must contain calendar rows through the printed wedding lookahead date. Extend the upstream calendar if that check fails. Future `NET_SALES` placeholders are not used as labels.


In [ ]:
from snowflake.snowpark.session import Session

if "session" not in globals() or session is None:
    sys.path.insert(0, str(UTILS_DIR))
    import Snowflake_configuration
    session = Session.builder.configs(Snowflake_configuration.ds1_role_json).create()

PREPARED_RUN_DIR = prepare_data(session)


## Review and handoff to modelling

Only use a run with `PREPARATION_COMPLETE.json` and manifest status `complete`.

- `local_train_data/`: observed history including the validation interval. The modelling notebook must split at `train_end`; this folder name does not mean all rows are training labels.
- `local_test_data/`: forecast dates, with missing targets and known covariates.
- `shared_calendar.parquet`: consistent calendar, including the optional pre-wedding feature.
- `column_roles.json`: exact ordered model feature lists and split dates.
- `saved_group_keys/group_keys.json` and `series_key_mapping.parquet`: selected population and original-to-prepared keys.
- `historical_sales_audit.parquet`: daily sales by model family/model for reconciling the selected population with your source and business totals.
- `preparation_manifest.json`: configuration, package versions and hashes; `chunk_plan.json`: deterministic dealer membership.

**For the next modelling notebook:** use this run directory as the data root; verify its completion marker and hashes; read the saved column order; build a fresh cache tied to this manifest. Fit the categorical encoder on the permitted training population, reserve a valid nonnegative unknown index, and assert that embeddings actually exist before training. Those steps belong to modelling and are deliberately not performed here.

The old modelling notebook must not be run unchanged against these new artifacts. This preparation also does not add SEGMENT, change target definitions, select new series, change loss weights, or establish that the business forecast target is achievable.

Snowflake API references: [Session.sql](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/snowpark/session), [temporary tables](https://docs.snowflake.com/en/user-guide/tables-temp-transient).


In [ ]:
manifest = json.loads((PREPARED_RUN_DIR / "preparation_manifest.json").read_text(encoding="utf-8"))
print("Status:", manifest["status"])
print("Series:", manifest["series_count"])
print("Dealers:", manifest["dealer_count"])
print("Prepared data root:", PREPARED_RUN_DIR.resolve())

preview = pd.read_parquet(PREPARED_RUN_DIR / "shared_calendar.parquet")
display(preview[[TIME_COL, WEDDING_COL, PRE_WEDDING_COL]].tail(21))

# Optional: close only when you are finished using this Snowflake session.
# session.close()
# session = None
